# PowerAI — Entrenar segmentacion del atleta

Entrena **yolo11m-seg.pt** para generar la mascara/silueta del atleta con bordes precisos.

## Dataset
- 3,859 imagenes etiquetadas con poligonos de segmentacion
- 1 clase: `athlete`
- Varios angulos, ejercicios y personas

## Antes de ejecutar
1. Crea un Dataset de Kaggle con `powerai_athlete_v1_corrected.zip`
2. Settings → GPU P100 → Turn on GPU
3. Add Input → tu dataset
4. Ejecuta celdas en orden (Shift+Enter)

In [ ]:
# 1) Instalar
!pip install -q ultralytics pyyaml

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"}')

In [ ]:
# 2) Extraer dataset y preparar train/val
import zipfile, shutil, yaml, random
from pathlib import Path

zip_path = None
for f in Path('/kaggle/input').rglob('*athlete*.zip'):
    zip_path = f; break

if zip_path is None:
    raise FileNotFoundError('powerai_athlete_v1_corrected.zip no encontrado. Subelo con Add Data.')

print(f'Dataset: {zip_path.name}')
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall('/kaggle/working/dataset')

ds = Path('/kaggle/working/dataset')
frames_dir = list(ds.rglob('frames'))[0]
masks_dir = list(ds.rglob('masks'))[0]

images = sorted([f for f in frames_dir.iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
paired = [img for img in images if (masks_dir / f'{img.stem}.txt').exists()]

random.seed(42)
random.shuffle(paired)
val_count = max(1, int(len(paired) * 0.2))
val_imgs = paired[:val_count]
train_imgs = paired[val_count:]

out = Path('/kaggle/working/athlete_dataset')
for split, imgs in [('train', train_imgs), ('val', val_imgs)]:
    img_dir = out / 'images' / split
    lbl_dir = out / 'labels' / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    for img in imgs:
        shutil.copy2(img, img_dir / img.name)
        mask = masks_dir / f'{img.stem}.txt'
        if mask.exists():
            shutil.copy2(mask, lbl_dir / f'{img.stem}.txt')

data_yaml = out / 'data.yaml'
data_yaml.write_text(yaml.dump({
    'path': '/kaggle/working/athlete_dataset',
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': ['athlete']
}))

print(f'Train: {len(train_imgs)}  |  Val: {len(val_imgs)}  |  Total: {len(paired)} mascaras')

In [ ]:
# 3) Entrenar segmentacion
# 120 epochs, patience=25, close_mosaic=15 para maxima calidad de mascara
from ultralytics import YOLO

model = YOLO('yolo11m-seg.pt')

model.train(
    data='/kaggle/working/athlete_dataset/data.yaml',
    epochs=120,
    imgsz=768,
    batch=4,
    device=0,
    project='/kaggle/working/runs',
    name='powerai_athlete_seg',
    exist_ok=True,
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    patience=25,
    close_mosaic=15,
    cos_lr=True,
)

print('\n*** Entrenamiento completado ***')

In [ ]:
# 4) Descargar modelo
import shutil
from pathlib import Path

best = Path('/kaggle/working/runs/powerai_athlete_seg/weights/best.pt')
output = Path('/kaggle/working/powerai_athlete_seg.pt')

if best.exists():
    shutil.copy2(best, output)
    size_mb = best.stat().st_size / 1e6
    print(f'Modelo: powerai_athlete_seg.pt ({size_mb:.1f} MB)')
    print('\nDescargalo del panel Output y copialo a: models/powerai_athlete_seg.pt')
else:
    print('Modelo no encontrado.')